## Plotting aligned parcellation maps across atlases

In [1]:
import os, pickle
import matplotlib.pyplot as plt
import matplotlib as mpl
from py_util_dx.py_utils import setProjectPath
from py_util_dx.data_utils import get_roi_vtx_from_fs32k, get_roi_pacels, get_glasser_labels
from scipy.stats import ttest_1samp
import Functional_Fusion.atlas_map as am 
import seaborn as sns 
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd 
import matplotlib as mpl 
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap
import nibabel as nib
from visualizations import plot_flatmap_labels
from nitools.cifti import surf_from_cifti
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import MDS
import matplotlib.colors as mcolors
from scipy.linalg import orthogonal_procrustes

In [2]:
mpl.rcParams['font.family'] = 'helvetica'   # Global font family
mpl.rcParams['font.size'] = 12              # Global font size 
full_width = 8.5
half_width = full_width/2

In [3]:
projectPath, mainResultsPath = setProjectPath()
dataset_name = 'MDTB' # or Demand
surface_helpers_dir = os.path.join(projectPath, 'surface_helpers')
resultsPath = os.path.join(mainResultsPath, 'START_A3_bayes_parcellation', f'{dataset_name}')

In [4]:
parcellations_of_interests = ['glasser', 'schaefer100']
Vs = []
n_parcels = [] 
roi = 'PFC' 
parcel_names = []

for parcellation in parcellations_of_interests:
    PKL_bayes_parcellation = os.path.join(resultsPath, parcellation, f'output_{roi}.pkl')
    with open(PKL_bayes_parcellation, 'rb') as pf:
        output = pickle.load(pf)
        V = output['V']
        Vs.append(V)
        n_parcels.append(V.shape[1])
        parcel_names.append(output['parcel_names_in_group']) 

In [5]:
Vs = np.hstack(Vs) 

In [ ]:
Vs.shape 

In [ ]:
n_parcels

In [ ]:
D = 1 - cosine_similarity(Vs.T)
D.shape 

In [9]:
mds = MDS(
    n_components=2,
    dissimilarity='precomputed',
    random_state=0,
    n_init=10,
    max_iter=300
)

X_joint = mds.fit_transform(D)  # (76, 2)

In [10]:
X_glasser = X_joint[:n_parcels[0]]
X_schaefer100 = X_joint[n_parcels[0]:] 

In [11]:
def embedding_to_rgb_hsv(X, S_max=0.85, V=0.9):
    x, y = X[:, 0], X[:, 1]

    hue = (np.arctan2(y, x) + np.pi) / (2 * np.pi)
    radius = np.sqrt(x**2 + y**2)

    saturation = np.clip(radius * S_max, 0, S_max)

    hsv = np.column_stack([hue, saturation, np.full(len(X), V)])
    rgb = mcolors.hsv_to_rgb(hsv)
    return rgb

In [12]:
color_rgb_glasser = embedding_to_rgb_hsv(X_glasser)
color_rgb_schaefer100 = embedding_to_rgb_hsv(X_schaefer100) 

In [ ]:
color_rgb_glasser.shape 

In [ ]:
plt.imshow(color_rgb_glasser[np.newaxis, :]) 

In [ ]:
plt.imshow(color_rgb_schaefer100[np.newaxis, :]) 

In [ ]:
color_df_glasser = pd.DataFrame(data=color_rgb_glasser, index=parcel_names[0], columns=['r', 'g', 'b']) 
color_df_glasser

In [ ]:
temp_df = pd.read_csv(os.path.join(surface_helpers_dir, f'atl-glasser.lut'), sep=' ', header=None)  
temp_df = temp_df.rename(columns={0:'indx', 1:'r', 2:'g', 3:'b', 4:'roi'})
temp_df

In [ ]:
temp_df.index = temp_df['roi'].str.lstrip('LR_').str.rstrip('_ROI') 
temp_df

In [ ]:
temp_df.update(color_df_glasser)
temp_df

In [20]:
temp_df.to_csv(os.path.join(surface_helpers_dir, f'atl-glasser.lut'), sep=' ', index=False, header=False)

In [4]:
# names = pd.read_csv(os.path.join(surface_helpers_dir, f'atl-schaefer100_no_alignment.lut'), header=None, skiprows=lambda x: x % 2 == 1)  
# names = names.rename(columns={0:'roi'})
# names 

,roi
0,17Networks_LH_VisCent_ExStr_1
1,17Networks_LH_VisCent_ExStr_2
2,17Networks_LH_VisCent_Striate_1
3,17Networks_LH_VisCent_ExStr_3
4,17Networks_LH_VisPeri_ExStrInf_1
...,...
95,17Networks_RH_DefaultC_Rsp_1
96,17Networks_RH_DefaultC_PHC_1
97,17Networks_RH_TempPar_1
98,17Networks_RH_TempPar_2


In [5]:
# colors = pd.read_csv(os.path.join(surface_helpers_dir, f'atl-schaefer100_no_alignment.lut'), header=None, skiprows=lambda x: x % 2 == 0, sep=' ')
# colors

,0,1,2,3,4
0,1,120,18,136,255
1,2,120,18,137,255
2,3,120,18,138,255
3,4,120,18,139,255
4,5,255,0,2,255
...,...,...,...,...,...
95,96,5,0,131,255
96,97,5,0,132,255
97,98,16,48,255,255
98,99,13,41,250,255


In [6]:
# temp_df = pd.merge(colors[[0, 1, 2, 3]], names, left_index=True, right_index=True) 
# temp_df

,0,1,2,3,roi
0,1,120,18,136,17Networks_LH_VisCent_ExStr_1
1,2,120,18,137,17Networks_LH_VisCent_ExStr_2
2,3,120,18,138,17Networks_LH_VisCent_Striate_1
3,4,120,18,139,17Networks_LH_VisCent_ExStr_3
4,5,255,0,2,17Networks_LH_VisPeri_ExStrInf_1
...,...,...,...,...,...
95,96,5,0,131,17Networks_RH_DefaultC_Rsp_1
96,97,5,0,132,17Networks_RH_DefaultC_PHC_1
97,98,16,48,255,17Networks_RH_TempPar_1
98,99,13,41,250,17Networks_RH_TempPar_2


In [7]:
# temp_df = temp_df.rename(columns={0: 'indx', 1: 'r', 2: 'g', 3: 'b'}) 
# temp_df

,indx,r,g,b,roi
0,1,120,18,136,17Networks_LH_VisCent_ExStr_1
1,2,120,18,137,17Networks_LH_VisCent_ExStr_2
2,3,120,18,138,17Networks_LH_VisCent_Striate_1
3,4,120,18,139,17Networks_LH_VisCent_ExStr_3
4,5,255,0,2,17Networks_LH_VisPeri_ExStrInf_1
...,...,...,...,...,...
95,96,5,0,131,17Networks_RH_DefaultC_Rsp_1
96,97,5,0,132,17Networks_RH_DefaultC_PHC_1
97,98,16,48,255,17Networks_RH_TempPar_1
98,99,13,41,250,17Networks_RH_TempPar_2


In [8]:
# temp_df[['r', 'g', 'b']] /= 255 
# temp_df 

,indx,r,g,b,roi
0,1,0.470588,0.070588,0.533333,17Networks_LH_VisCent_ExStr_1
1,2,0.470588,0.070588,0.537255,17Networks_LH_VisCent_ExStr_2
2,3,0.470588,0.070588,0.541176,17Networks_LH_VisCent_Striate_1
3,4,0.470588,0.070588,0.545098,17Networks_LH_VisCent_ExStr_3
4,5,1.000000,0.000000,0.007843,17Networks_LH_VisPeri_ExStrInf_1
...,...,...,...,...,...
95,96,0.019608,0.000000,0.513725,17Networks_RH_DefaultC_Rsp_1
96,97,0.019608,0.000000,0.517647,17Networks_RH_DefaultC_PHC_1
97,98,0.062745,0.188235,1.000000,17Networks_RH_TempPar_1
98,99,0.050980,0.160784,0.980392,17Networks_RH_TempPar_2


In [9]:
# temp_df = temp_df.set_index('indx') 
# temp_df

,r,g,b,roi
indx,,,,
1,0.470588,0.070588,0.533333,17Networks_LH_VisCent_ExStr_1
2,0.470588,0.070588,0.537255,17Networks_LH_VisCent_ExStr_2
3,0.470588,0.070588,0.541176,17Networks_LH_VisCent_Striate_1
4,0.470588,0.070588,0.545098,17Networks_LH_VisCent_ExStr_3
5,1.000000,0.000000,0.007843,17Networks_LH_VisPeri_ExStrInf_1
...,...,...,...,...
96,0.019608,0.000000,0.513725,17Networks_RH_DefaultC_Rsp_1
97,0.019608,0.000000,0.517647,17Networks_RH_DefaultC_PHC_1
98,0.062745,0.188235,1.000000,17Networks_RH_TempPar_1


In [12]:
temp_df = pd.read_csv(os.path.join(surface_helpers_dir, f'atl-schaefer100.lut'), sep='\s+', header=None, names=['indx', 'r', 'g', 'b', 'roi'])
temp_df 

,indx,r,g,b,roi
0,1,0.470588,0.070588,0.533333,17Networks_LH_VisCent_ExStr_1
1,2,0.470588,0.070588,0.537255,17Networks_LH_VisCent_ExStr_2
2,3,0.470588,0.070588,0.541176,17Networks_LH_VisCent_Striate_1
3,4,0.470588,0.070588,0.545098,17Networks_LH_VisCent_ExStr_3
4,5,1.000000,0.000000,0.007843,17Networks_LH_VisPeri_ExStrInf_1
...,...,...,...,...,...
95,96,0.019608,0.000000,0.513725,17Networks_RH_DefaultC_Rsp_1
96,97,0.019608,0.000000,0.517647,17Networks_RH_DefaultC_PHC_1
97,98,0.062745,0.188235,1.000000,17Networks_RH_TempPar_1
98,99,0.050980,0.160784,0.980392,17Networks_RH_TempPar_2


In [ ]:
temp_df = temp_df.set_index('indx') 
temp_df 

In [ ]:
temp_df[0:20]

In [ ]:
color_df_schaefer100 = pd.DataFrame(data=color_rgb_schaefer100, index=list(map(int, parcel_names[1])), columns=['r', 'g', 'b']) 
color_df_schaefer100.index.name = 'indx' 
color_df_schaefer100

In [ ]:
temp_df.update(color_df_schaefer100)
temp_df[0:20]

In [ ]:
color_df_schaefer100.index 

In [ ]:
temp_df.index 

In [11]:
temp_df.to_csv(os.path.join(surface_helpers_dir, f'atl-schaefer100.lut'), sep=' ', index=True, header=False) 

In [ ]:
# # getting the mds solutions for individuals 
# Vs_indv = [] 

# for parcellation in parcellations_of_interests:
#     PKL_Vs = os.path.join(mainResultsPath, 'START_A4_parcellation_evaluation', dataset_name, parcellation, f'output_{roi}_cv.pkl')
#     with open(PKL_Vs, 'rb') as pf:
#         output_indv = pickle.load(pf)
#         Vs_indv.append(output_indv['Vs'])

# Vs_indv = np.concatenate(Vs_indv, axis=-1) 
# Vs_indv.shape 

In [28]:
# n_subjects = Vs_indv.shape[0]

In [ ]:
# Ds = [] 
# for subjI in np.arange(n_subjects):
#     Ds.append(1 - cosine_similarity(Vs_indv[subjI].T))

# Ds = np.array(Ds) 
# Ds[0].shape 

In [ ]:
# X_indiv = [mds.fit_transform(D) for D in Ds] 
# X_indiv = np.array(X_indiv) 
# X_indiv.shape 

In [ ]:
# X_indiv_glasser = X_indiv[:, :n_parcels[0]]
# X_indiv_schaefer100 = X_indiv[:, n_parcels[0]:] 
# X_indiv_glasser.shape 

In [40]:
# for subjI in np.arange(n_subjects):
#     R, _ = orthogonal_procrustes(X_indiv_glasser[subjI], X_glasser)
#     X_indiv_glasser[subjI] = X_indiv_glasser[subjI] @ R 

#     R, _ = orthogonal_procrustes(X_indiv_schaefer100[subjI], X_schaefer100)
#     X_indiv_schaefer100[subjI] = X_indiv_schaefer100[subjI] @ R 

In [41]:
# color_rgb_glasser_indiv = [embedding_to_rgb_hsv(y) for y in X_indiv_glasser] 
# color_rgb_schaefer100_indiv = [embedding_to_rgb_hsv(y) for y in X_indiv_schaefer100] 

In [ ]:
# # generating color for each subject for the individualized glass atlas
# temp_df = pd.read_csv(os.path.join(surface_helpers_dir, f'atl-glasser.lut'), sep=' ', header=None)  
# temp_df = temp_df.rename(columns={0:'indx', 1:'r', 2:'g', 3:'b', 4:'roi'})
# temp_df.index = temp_df['roi'].str.lstrip('LR_').str.rstrip('_ROI') 
# temp_df

In [43]:
# for subjI in np.arange(n_subjects):
#     color_df = pd.DataFrame(data=color_rgb_glasser_indiv[subjI], index=parcel_names[0], columns=['r', 'g', 'b']) 
#     temp_df.update(color_df)
#     temp_df.to_csv(os.path.join(surface_helpers_dir, f'atl-glasser-aligned_subj{subjI}.lut'), sep=' ', index=False, header=False) 

In [ ]:
# # generating color for each subject for the individualized schaefer100 atlas
# temp_df = pd.read_csv(os.path.join(surface_helpers_dir, f'atl-schaefer100.lut'), sep='\s+', header=None, names=['indx', 'r', 'g', 'b', 'roi'])
# temp_df = temp_df.set_index('indx') 
# temp_df 

In [45]:
# for subjI in np.arange(n_subjects):
#     color_df = pd.DataFrame(data=color_rgb_schaefer100_indiv[subjI], index=list(map(int, parcel_names[1])), columns=['r', 'g', 'b']) 
#     color_df.index.name = 'indx' 
#     temp_df.update(color_df)
#     temp_df.to_csv(os.path.join(surface_helpers_dir, f'atl-schaefer100-aligned_subj{subjI}.lut'), sep=' ', index=True, header=False) 

In [52]:
# atlas_name = 'glasser'
atlas_name = 'schaefer100'
# atlas_name = 'yeo17'
surface_helpers_dir = os.path.join(projectPath, 'surface_helpers')
resultsPath = os.path.join(mainResultsPath, 'START_A4_parcellation_evaluation', f'{dataset_name}', atlas_name, 'aligned_atlas')
if not os.path.exists(resultsPath):
    os.makedirs(resultsPath)

In [53]:
example_subjIs = np.arange(n_subjects)
n_example_subjects = len(example_subjIs) 
frames_dict = {
    'PFC': [[-210, 25, -90, 140], [0, 240, -110, 120]], 
    'visual': [[100, 230, -70, 160], [-250, -50, -70, 130]], 
    'somatosensory':[[-10, 80, -10, 160], [-60, 30, -10, 140]], 
    'parietal': [[20, 140, 0, 160], [-140, -30, 0, 160]]
}
gridspec_dict = {
    'PFC': [-0.2, -0.05],
    'visual': [-0.2, -0.12],
    'parietal': [-0.55, -0.25],
    'somatosensory': [-0.55, -0.05]
}

In [54]:
atlas_str = 'fs32k'
atlas, ainf = am.get_atlas(atlas_str)

In [55]:
PKL_individualized_parcellation = os.path.join(projectPath, 'results', 'START_A3_bayes_parcellation', f'{dataset_name}', atlas_name, f'output_{roi}.pkl')
with open(PKL_individualized_parcellation, 'rb') as pf:
    output_indiv = pickle.load(pf)
    labels_in_group = output_indiv['labels_in_group']
    parcel_names_in_group = output_indiv['parcel_names_in_group']

    U_individual = output_indiv['U_individual']     # data only parcellation
    indiv_parcellation = output_indiv['indiv_parcellation']

    U_group = output_indiv['U_group']
    group_parcellation = output_indiv['group_parcellation']

    group_parcellation[group_parcellation==0] = np.max(group_parcellation)

In [ ]:
# plotting the group map
# fig, ax = plt.subplots(1, 2, figsize=(full_width, 8), constrained_layout=True, gridspec_kw={'wspace': gridspec_dict[roi][0], 'hspace': gridspec_dict[roi][1]})
fig, ax = plt.subplots(1, 2, figsize=(full_width, 8), constrained_layout=True)
fig.set_figwidth(full_width)

[label_L, label_R] = surf_from_cifti(atlas.data_to_cifti(group_parcellation.reshape(1, -1)))
plt.axes(ax[0])
plot_flatmap_labels(label_L, 'L', frame=frames_dict[roi][0], borders=None, atlas=atlas_name)

plt.axes(ax[1])
plot_flatmap_labels(label_R, 'R', frame=frames_dict[roi][1], borders=None, atlas=atlas_name)

plt.suptitle(f'group {atlas_name}, {roi}')
plt.tight_layout() 

JPG_fig = os.path.join(resultsPath, f'group_{atlas_name}_{roi}.jpg')
plt.savefig(JPG_fig, dpi=500, format='jpg')

In [ ]:
for subjI in example_subjIs:
    fig, ax = plt.subplots(1, 2, figsize=(full_width, 8), constrained_layout=True, gridspec_kw={'wspace': gridspec_dict[roi][0], 'hspace': gridspec_dict[roi][1]})
    fig.set_figwidth(full_width)
    
    [label_L, label_R] = surf_from_cifti(atlas.data_to_cifti(indiv_parcellation[subjI].reshape(1, -1)))
    plt.axes(ax[0])
    plot_flatmap_labels(label_L, 'L', frame=frames_dict[roi][0], borders=None, atlas=atlas_name, lut=os.path.join(surface_helpers_dir, f'atl-{atlas_name}-aligned_subj{subjI}.lut'))
    
    plt.axes(ax[1])
    plot_flatmap_labels(label_R, 'R', frame=frames_dict[roi][1], borders=None, atlas=atlas_name, lut=os.path.join(surface_helpers_dir, f'atl-{atlas_name}-aligned_subj{subjI}.lut'))

    plt.suptitle(f'subject {subjI}, {roi}')
    plt.tight_layout() 

    JPG_fig = os.path.join(resultsPath, f'example_{atlas_name}_{roi}_subject_{subjI}.jpg')
    plt.savefig(JPG_fig, dpi=500, format='jpg')